# Три алгоритма на NumPy

Маленькие примеры помогают проверить не только результат, но и ход вычислений.
Исходники в `ml_scratch/` используют только NumPy и стандартную библиотеку.
Разбиения и эталонные модели в примерах берём из scikit-learn.

In [1]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml_scratch/knn.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ml_scratch import KNNClassifier, LinearRegressionGD, DecisionTreeClassifier

## 1. kNN: расстояния и голоса

У kNN нет подбираемых градиентом весов: `fit` сохраняет обучающие объекты.
Для новой точки считаем евклидовы расстояния, выбираем k ближайших и голосуем.
Доля голосов даёт predict_proba. Масштаб признаков влияет на расстояние.

In [2]:
X = np.array([[0.0], [1.0], [4.0], [5.0], [6.0]])
y = np.array(['left', 'left', 'right', 'right', 'right'])
query = np.array([[3.0]])
knn = KNNClassifier(n_neighbors=3).fit(X, y)
distances = np.sqrt(((X - query[0]) ** 2).sum(axis=1))
neighbors = np.argsort(distances, kind='stable')[:3]
print('Distances:', distances)
print('Neighbors:', neighbors, 'labels:', y[neighbors])
print('Classes:', knn.classes_)
print('Probability:', knn.predict_proba(query))
print('Prediction:', knn.predict(query))

Distances: [3. 2. 1. 2. 3.]
Neighbors: [2 1 3] labels: ['right' 'left' 'right']
Classes: ['left' 'right']
Probability: [[0.33333333 0.66666667]]
Prediction: ['right']


В этом примере побеждает right: два голоса из трёх. При ничьей голосов
выбирается первый класс в `np.unique`. Равные расстояния разрешаются порядком train.

## 2. Линейная регрессия: уменьшаем ошибку

Модель `y_hat = X @ w + b`. Цель — MSE/2 + l2*||w||²/2.
Градиент по весам: `X.T @ (y_hat - y) / n + l2 * w`; по свободному члену —
средняя ошибка. Обновляем параметры против градиента с шагом learning_rate.

In [3]:
X = np.linspace(-2, 2, 100).reshape(-1, 1)
y = 3 * X[:, 0] + 4
linear = LinearRegressionGD(learning_rate=0.05, tol=1e-10).fit(X, y)
print('Expected w=3, b=4')
print('Found:', linear.coef_, linear.intercept_)
print('Loss before/after:', linear.loss_history_[0], linear.loss_history_[-1])
print('Iterations:', linear.n_iter_)
np.testing.assert_allclose(linear.predict([[2.0]]), [10.0], atol=1e-7)

Expected w=3, b=4
Found: [3.] 3.999999999900346
Loss before/after: 14.121212121212123 4.965464613791882e-21
Iterations: 476


L2 уменьшает модуль весов, но не штрафует b. Это может уменьшить переобучение,
а слишком сильная регуляризация приводит к недообучению. Ниже меняем только L2.

In [4]:
for l2 in [0.0, 0.1, 1.0]:
    model = LinearRegressionGD(l2=l2).fit(X, y)
    print(f'l2={l2}: w={model.coef_[0]:.4f}, b={model.intercept_:.4f}')

l2=0.0: w=3.0000, b=4.0000
l2=0.1: w=2.7946, b=4.0000
l2=1.0: w=1.7290, b=4.0000


## 3. Дерево: делим по порогам

Gini = 1 - sum(p_class²). Перебираем признаки и пороги между соседними различными
значениями, выбираем максимальное уменьшение взвешенного Gini и строим детей.
В листе сохраняем доли классов. Глубина корня — ноль.

In [5]:
X_tree = np.array([[0], [1], [2], [3]])
y_tree = np.array([0, 0, 1, 1])
tree = DecisionTreeClassifier(max_depth=1).fit(X_tree, y_tree)
print('Root Gini:', tree.root_.impurity)
print('Feature:', tree.root_.feature, 'threshold:', tree.root_.threshold)
print('Leaf probabilities:', tree.root_.left.probabilities, tree.root_.right.probabilities)
print('Predictions:', tree.predict(X_tree))
np.testing.assert_array_equal(tree.predict(X_tree), y_tree)

Root Gini: 0.5
Feature: 0 threshold: 1.5
Leaf probabilities: [1. 0.] [0. 1.]
Predictions: [0 0 1 1]


## 4. Сравнение kNN с библиотекой

Train/test разделены до масштабирования. StandardScaler учится только на train.
Значение k=5 фиксировано заранее; этот пример проверяет совпадение реализаций.
Полное сравнение и train CV по гиперпараметрам выполняются командой
`python -m examples.compare_models` и сохранены в reports.

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
own = KNNClassifier(5).fit(X_train, y_train)
reference = KNeighborsClassifier(5).fit(X_train, y_train)
np.testing.assert_allclose(own.predict_proba(X_test), reference.predict_proba(X_test))
print('Agreement:', np.mean(own.predict(X_test) == reference.predict(X_test)))
print('Test accuracy:', np.mean(own.predict(X_test) == y_test))

Agreement: 1.0
Test accuracy: 0.9210526315789473


## Проверка понимания

- Почему масштабирование нужно kNN? Крупные числовые значения иначе доминируют в расстоянии.
- Что меняет learning_rate? Величину шага обновления; слишком большой шаг может вызвать расходимость.
- Зачем ограничивать дерево? Большая глубина повышает способность подстроиться под train и шум.
- Можно ли подобрать k на test? Нет, для настройки используем train CV.
- Гарантирует ли совпадение на одном наборе полную эквивалентность sklearn? Нет: важны
  крайние случаи, правила ничьих, точность вычислений и другие ограничения реализации.

Следующий шаг: вычисли одно обновление GD и одно Gini-разбиение вручную,
затем найди эти вычисления в коде и тестах.